# Investigate missing grains across scanning 3DXRD layers

Load layer datasets and their indexed grains, display their COMs as a 3D stack, and use nearby-layer COMs and UBIs as seeds in each target layer. Search **both residual 4D peaks and residual 2D peaks independently**: a grain need not have survived 4D merging to be investigated.

Coordinates follow `tomo_2_map.ipynb`: grain translation `[sx, sy, 0]`, and `dty = y0 - sx*sin(omega) - sy*cos(omega)`. Layer `z` is used for stack placement and neighbour selection, **not** as a detector geometry translation. All layers must share a sample frame, omega zero/sign, phase, and physical units. Supply registered COMs if the sample moved between scans. COM propagation is an initial guess; grain shape and COM can change with height.

Results are screening candidates, not proof of a missing grain. Grain termination, twins, overlapping reflections, and geometry errors require review. This notebook never changes the input datasets, peak lists, or grain files. Defaults below are exploratory thresholds, not calibrated acceptance criteria.

In [ ]:
import os
os.environ.setdefault('OMP_NUM_THREADS', '1')
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation
from scipy.optimize import least_squares
from IPython.display import display
from ImageD11 import grain, columnfile
from ImageD11.sinograms import dataset, geometry
from ImageD11.sinograms.sinogram import GrainSinogram
from tqdm.auto import tqdm

## Inputs and controls

The example is configured for your two `pureIron_2_heating_3_sff_after_20min` layers. Both use **millimetres**, with dty steps approximately 0.002 mm; phase `Fe` has space group 229. Stack heights are read automatically from `<scan>/instrument/positioners/difftz` in each dataset's `masterfile`, using its recorded scans. The notebook sorts layers by these motor positions and displays the derived `layer_z_mm`; no manual heights are required. Motor order does not establish the sample-up direction. You can add more entries to `layers` in any order. Motor heights, COMs, `y0`, and distance controls must use the same units as `dty` (mm for these data). Each dataset loads its own calibration and phase. Grains come from `ds.grainsfile`, or an optional `.ubi` grain file with **sample-frame COM translations**. A layer with no indexed grains can use `grains=[]`.

Optional `remaining_2d` / `remaining_4d` paths are ImageD11 columnfiles (HDF5 or text), already containing **only remaining peaks**, with corrected `sc,fc` (or `xc,yc`), `omega,dty` and `sum_intensity`. They are treated as your authoritative residual lists. With no paths, 4D peaks and raw 2D peaks come from the dataset; a peak is considered explained only if an existing grain matches both its orientation and COM trajectory. This fallback uses the current phase's grain list, not other phases or saved fractional ownership; supply authoritative remaining lists when those matter.

Set crystal symmetry explicitly: SciPy groups `O` (cubic), `D6` (hexagonal), `D4` (tetragonal), `D3`, `D2`, `C2`, `C1`. Non-cubic groups must have axes aligned with your ImageD11 crystal Cartesian frame; alternatively supply `symmetry_matrices` explicitly. No symmetry is inferred from the lattice centring symbol.

A single `trajectory_width` defines a local band including grain extent and beam width. Set it wide enough for those, but narrow enough to reject neighbouring grains. The COM grid permits shifts within `search_radius`; increase cautiously. Thresholds only flag candidates for review.

For these inputs the existing grain refinements include fractional peak ownership. The geometric residual fallback below does **not** subtract those saved fractions: it excludes peaks matching the current grain COM/orientation model. Its output is a diagnostic residual pool, not an exact balance of unassigned intensity. Use explicit remaining lists when you need that distinction. The search widths are starting settings expressed in mm; tune them against the grain trajectory plots, especially for large grains.

In [ ]:
phase_str = 'Fe'
data_root = '/data/visitor/ma7056/id11/20260624/PROCESSED_DATA/pureIron_2_heating_3'
layer_names = ['pureIron_2_heating_3_sff_after_20min0',
               'pureIron_2_heating_3_sff_after_20min1']
layer_z_motor = 'difftz'  # read automatically from each dataset's raw master file
layers = [dict(name=name,
               dataset=os.path.join(data_root, name, name + '_dataset.h5'))
          for name in layer_names]
# No separate residual columnfiles were found in these directories.
# Omitted remaining_2d/remaining_4d keys derive residuals from the full peak pools.
# To use authoritative residual lists, add the corresponding paths per layer.
# Optional per-layer keys: y0=..., grains='/path/grains.ubi', or grains=[]
symmetry_group = 'O'  # Fe phase: space group 229, cubic
symmetry_matrices = None
neighbour_layers = 1  # search seeds from this many loaded layers on either side
max_z_gap = None      # optionally limit absolute source-to-target z distance
match_distance = 0.010  # mm; exploratory COM matching radius (5 dty steps)
match_angle_deg = 1.0
trajectory_width = 0.006  # mm; initial local half-width (3 dty steps)
search_radius = 0.010  # mm; initial COM search radius (5 dty steps)
search_grid_points = 5  # odd; covers +/- search_radius in sx and sy
hkl_tol = 0.05         # Euclidean distance to nearest integer hkl
max_refine_angle_deg = 1.0
max_cell_relative_change = 0.01
min_unique_hkl = 8
min_spots = 12
min_omega_bins = 4
omega_bin_deg = 10.0
chunk_size = 100000
plot_peak_limit = 20000

## Load the grain COM stack

Missing or non-finite grain COMs are an error: use COMs from the mapping stage, not orientation-only indexing output. The stack contains grain COM points, not a segmented voxel volume. Grain IDs below are local row indices and are never assumed to match between layers.

The height motor must be finite and stationary within each recorded scan and agree across scans in a layer (absolute tolerance 1e-9 mm). Missing or changing heights raise an error rather than inventing a layer spacing.

In [ ]:
def layer_motor_z_mm(ds):
    # ds.scans can contain many slices of the same scan (e.g. 1.1::[0:720]).
    scans = sorted(set(str(scan).split('::', 1)[0] for scan in ds.scans))
    if not scans:
        raise ValueError('No scan metadata available to determine layer height.')
    positions = []
    with h5py.File(ds.masterfile, 'r') as master:
        for scan in scans:
            path = scan + '/instrument/positioners/' + layer_z_motor
            if path not in master:
                raise ValueError('Missing height motor {} in {}'.format(path, ds.masterfile))
            values = np.asarray(master[path][()], dtype=float).ravel()
            if not len(values) or not np.isfinite(values).all():
                raise ValueError('Invalid height motor values at ' + path)
            # A moving height motor does not describe a single layer.
            if not np.allclose(values, values[0], rtol=0, atol=1e-9):
                raise ValueError('Height motor varies during scan: ' + path)
            positions.append(float(values[0]))
    if not np.allclose(positions, positions[0], rtol=0, atol=1e-9):
        raise ValueError('Height motor differs between scans in ' + ds.masterfile)
    return positions[0]

def misorientation_deg(u, v):
    traces = np.array([np.trace(np.dot(np.dot(u, s), v.T)) for s in symops])
    return np.degrees(np.arccos(np.clip((traces.max() - 1.0) / 2.0, -1, 1)))

def same_grain(g, xy, other):
    return (np.linalg.norm(np.asarray(other.translation)[:2] - xy) <= match_distance
            and misorientation_deg(g.U, other.U) <= match_angle_deg)

def load_stack(specs):
    if len(specs) < 2:
        raise ValueError('Configure at least two layers in the input cell.')
    if len(set(s['name'] for s in specs)) != len(specs):
        raise ValueError('Layer names must be unique.')
    loaded = []
    for input_spec in specs:
        spec = dict(input_spec)
        ds = dataset.load(spec['dataset'])
        spec['z'] = layer_motor_z_mm(ds)
        ds.phases = ds.get_phases_from_disk()
        uc = ds.phases.unitcells[phase_str]
        gs = spec.get('grains')
        if gs is None:
            gs = ds.get_grains_from_disk(phase_str)
        elif isinstance(gs, str):
            gs = grain.read_grain_file(gs)
        for gid, g in enumerate(gs):
            if g.translation is None or not np.isfinite(g.translation[:2]).all():
                raise ValueError('{} grain {} has no finite COM'.format(spec['name'], gid))
        y0 = spec.get('y0', getattr(ds, 'y0', None))
        if y0 is None or not np.isfinite(y0):
            raise ValueError('Supply calibrated y0 for ' + spec['name'])
        loaded.append(dict(spec=spec, ds=ds, grains=gs, y0=float(y0), uc=uc))
    zs = np.asarray([layer['spec']['z'] for layer in loaded])
    if len(np.unique(zs)) != len(zs):
        raise ValueError('Layers have identical height motor positions.')
    return sorted(loaded, key=lambda layer: layer['spec']['z'])

symops = (Rotation.create_group(symmetry_group).as_matrix()
          if symmetry_matrices is None else np.asarray(symmetry_matrices, dtype=float))
if (symops.ndim != 3 or symops.shape[1:] != (3, 3)
        or not np.allclose(np.matmul(symops, symops.transpose(0, 2, 1)), np.eye(3))
        or not np.allclose(np.linalg.det(symops), 1)):
    raise ValueError('Supply proper orthogonal crystal symmetry matrices.')
if (trajectory_width <= 0 or search_radius <= 0 or match_distance <= 0
        or not 0 < hkl_tol < 0.5 or not 0 < omega_bin_deg <= 180
        or search_grid_points < 3 or search_grid_points % 2 != 1
        or chunk_size < 1 or neighbour_layers < 1):
    raise ValueError('Check positive widths, grid, chunk and tolerance controls.')
stack = load_stack(layers)
print('Loaded:', [(l['spec']['name'], len(l['grains'])) for l in stack])
# Derived values in stack order, not manual inputs.
layer_z_mm = [layer['spec']['z'] for layer in stack]
display(pd.DataFrame([dict(layer=layer['spec']['name'], motor=layer_z_motor,
                           layer_z_mm=layer['spec']['z']) for layer in stack]))

In [ ]:
def plot_stack(stack, candidates=None):
    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')
    for layer in stack:
        xy = np.array([g.translation[:2] for g in layer['grains']]).reshape(-1, 2)
        ax.scatter(xy[:, 0], xy[:, 1], np.full(len(xy), layer['spec']['z']),
                   s=12, label=layer['spec']['name'])
    for c in candidates or []:
        if c['review_candidate']:
            z = stack[c['target']]['spec']['z']
            ax.scatter(c['xy'][0], c['xy'][1], z, marker='x', c='red', s=60)
    ax.set(xlabel='Sample sx (dty units)', ylabel='Sample sy (dty units)',
           zlabel='Layer z (dty units)', title='Indexed COM stack; red crosses: candidates')
    ax.legend()
    plt.show()

plot_stack(stack)

## Propose absent neighbours

For each target, propagate every grain from the configured nearby layers. Skip it when a target grain has both a matching COM and symmetry-equivalent orientation. Merge equivalent seeds arriving from either side, keeping their provenance. Distinct nearby orientations remain separate hypotheses. This permits recovery at stack edges and in layers with no indexed grains. It does not extrapolate beyond `neighbour_layers` or through unregistered frame changes.

In [ ]:
def propose_seeds(stack):
    seeds = []
    for ti, target in enumerate(stack):
        local = []
        for si in sorted(range(len(stack)), key=lambda i: abs(stack[i]['spec']['z'] - target['spec']['z'])):
            source = stack[si]
            if si == ti or abs(si - ti) > neighbour_layers:
                continue
            if max_z_gap is not None and abs(source['spec']['z'] - target['spec']['z']) > max_z_gap:
                continue
            for gid, g in enumerate(source['grains']):
                xy = np.asarray(g.translation[:2], dtype=float)
                if any(same_grain(g, xy, other) for other in target['grains']):
                    continue
                provenance = (source['spec']['name'], gid)
                duplicate = next((s for s in local
                                  if np.linalg.norm(s['xy'] - xy) <= match_distance
                                  and misorientation_deg(s['grain'].U, g.U) <= match_angle_deg), None)
                if duplicate is not None:
                    duplicate['sources'].append(provenance)
                else:
                    local.append(dict(target=ti, xy=xy.copy(), grain=g, sources=[provenance]))
        seeds.extend(local)
    return seeds

seeds = propose_seeds(stack)
print('Missing-grain hypotheses:', len(seeds))
display(pd.DataFrame([dict(target=stack[s['target']]['spec']['name'],
                           sx=s['xy'][0], sy=s['xy'][1], sources=s['sources']) for s in seeds]))

## Peak pools and local searches

Peak geometry follows the existing scanning notebooks: each layer's calibration at its configured reference translation; sample COM enters the scanning trajectory. The code does not pass the stack height to `updateGeometry`.

Process one target layer at a time. Raw 2D peaks are spatially corrected only in chunks, and only local rows are retained per hypothesis. The dataset's raw `pk2d` dictionary itself may still be large. Explicit remaining columnfiles are loaded whole by ImageD11. Temporary masks cost about one byte per chunk row; local subsets and g-vectors cost more and grow with the number of peaks near a trajectory. No full peak-by-grain matrix is allocated.

Without supplied residual files, all known grains are scored once per chunk, using orientation AND spatial support; this is conservative but can be expensive. Zero hkl is rejected. No assumption is made about frame ordering or uniform omega spacing.

In [ ]:
def hkl_match(cf, ubi):
    gv = np.column_stack((cf.gx, cf.gy, cf.gz))
    real = np.dot(gv, ubi.T)
    integer = np.rint(real)
    error = np.linalg.norm(real - integer, axis=1)
    good = np.isfinite(error) & (error < hkl_tol) & np.any(integer != 0, axis=1)
    return good, integer, error

def trajectory(xy, omega, y0):
    return geometry.dty_values_grain_in_beam(xy[0], xy[1], y0, omega)

def known_support(cf, layer):
    explained = np.zeros(cf.nrows, dtype=bool)
    for g in layer['grains']:
        near = np.abs(cf.dty - trajectory(g.translation[:2], cf.omega, layer['y0'])) <= trajectory_width
        if near.any():
            explained |= near & hkl_match(cf, g.ubi)[0]
    return explained

def iter_pool(layer, kind):
    path = layer['spec'].get('remaining_' + kind)
    ds = layer['ds']
    raw = kind == '2d' and path is None
    pool = ds.pk2d if raw else (columnfile.columnfile(path) if path else ds.get_cf_4d())
    count = len(pool['omega']) if raw else pool.nrows
    corrector = ds.get_spatial_corrector() if raw else None
    for start in range(0, count, chunk_size):
        stop = min(start + chunk_size, count)
        if raw:
            cf = ds.get_colfile_from_peaks_dict(
                {key: values[start:stop] for key, values in pool.items()}, corrector=corrector)
        else:
            cf = pool.copyrows(np.arange(start, stop))
        ds.update_colfile_pars(cf, phase_name=phase_str)
        required = ('omega', 'dty', 'sum_intensity', 'gx', 'gy', 'gz')
        if any(key not in cf.titles for key in required):
            raise ValueError('Missing peak columns in ' + kind)
        rows = np.arange(start, stop, dtype=np.int64)
        finite = np.isfinite(np.column_stack([cf[k] for k in required])).all(axis=1)
        keep = finite if path else finite & ~known_support(cf, layer)
        yield cf.copyrows(keep), rows[keep]

def local_peak_sets(layer, local_seeds, kind):
    # Row indices always refer to the input pool of this kind, never the other kind.
    pieces = [[] for s in local_seeds]
    for cf, rows in iter_pool(layer, kind):
        for j, seed in enumerate(local_seeds):
            broad_width = trajectory_width + np.sqrt(2.0) * search_radius
            near = np.abs(cf.dty - trajectory(seed['xy'], cf.omega, layer['y0'])) <= broad_width
            if near.any():
                sub = cf.copyrows(near)
                # Looser seed gate permits a small orientation refinement later.
                real = np.dot(np.column_stack((sub.gx, sub.gy, sub.gz)), seed['grain'].ubi.T)
                gate = ((np.linalg.norm(real - np.rint(real), axis=1) < 2 * hkl_tol)
                        & np.any(np.rint(real) != 0, axis=1))
                if gate.any():
                    names = ['omega', 'dty', 'sum_intensity', 'gx', 'gy', 'gz', 'eta', 'tth']
                    if 'spot3d_id' in sub.titles:
                        names.append('spot3d_id')
                    data = {key: sub[key][gate].copy() for key in names}
                    data['pool_row'] = rows[near][gate]
                    pieces[j].append(data)
    result = []
    for parts in pieces:
        if not parts:
            result.append(None)
        else:
            result.append({key: np.concatenate([p[key] for p in parts]) for key in parts[0]})
    return result

## Fit COM and test reflection support

Search a bounded COM grid for each seed, then fit its sinusoidal trajectory with a robust loss and fixed calibrated `y0`. Repeated 2D observations of one `spot3d_id` share weight. A broad spatial band is needed for extended grains; the fitted centre is an estimate, not a reconstructed COM.

Refine UBI by linear least squares only when at least three independent integer hkl vectors exist, averaging repeated observations of each hkl. Reject updates that exceed orientation or cell-change bounds. There is no free d0 correction. Report support at the original seed **and** after refinement.

4D and 2D candidates are retained separately. Counts from these pools are correlated and are never added together as independent evidence. Several candidates can claim the same residual peaks: the later overlap report exposes this ambiguity.

In [ ]:
def as_colfile(data):
    return columnfile.colfile_from_dict(data)

def support(data, xy, ubi, y0):
    if data is None:
        return np.zeros(0, dtype=bool), np.zeros((0, 3)), np.zeros(0)
    good, integer, err = hkl_match(as_colfile(data), ubi)
    return good & (np.abs(data['dty'] - trajectory(xy, data['omega'], y0)) <= trajectory_width), integer, err

def fit_candidate(seed, data, layer, kind):
    xy0 = seed['xy']
    ubi0 = seed['grain'].ubi.copy()
    xy, ubi = xy0.copy(), ubi0.copy()
    initial = support(data, xy0, ubi0, layer['y0'])[0].sum()
    if data is not None:
        offsets = np.linspace(-search_radius, search_radius, search_grid_points)
        # Rank by distinct spots when available, so long 2D tracks do not dominate.
        def count_at(pos):
            keep = support(data, pos, ubi, layer['y0'])[0]
            return (len(np.unique(data['spot3d_id'][keep]))
                    if 'spot3d_id' in data else int(keep.sum()))
        positions = [xy0 + [dx, dy] for dx in offsets for dy in offsets]
        positions.insert(0, xy0.copy())  # ties retain the propagated COM
        xy = max(positions, key=count_at).copy()
        for iteration in range(3):
            keep, hkls, _ = support(data, xy, ubi, layer['y0'])
            if keep.sum() < 3:
                break
            om, dt = data['omega'][keep], data['dty'][keep]
            design = np.column_stack((np.sin(np.radians(om)), np.cos(np.radians(om))))
            if np.linalg.matrix_rank(design) < 2:
                break
            weights = np.ones(len(om))
            if 'spot3d_id' in data:
                _, inverse, counts = np.unique(data['spot3d_id'][keep], return_inverse=True, return_counts=True)
                weights = 1.0 / np.sqrt(counts[inverse])
            fit = least_squares(lambda p: weights * (trajectory(p, om, layer['y0']) - dt),
                                xy, bounds=(xy0 - search_radius, xy0 + search_radius),
                                loss='soft_l1', f_scale=trajectory_width / 2)
            xy = fit.x
            keep, hkls, _ = support(data, xy, ubi, layer['y0'])
            unique, inverse = np.unique(hkls[keep], axis=0, return_inverse=True)
            if len(unique) < 3 or np.linalg.matrix_rank(unique) < 3:
                continue
            gv = np.column_stack((data['gx'], data['gy'], data['gz']))[keep]
            gv_mean = np.array([gv[inverse == k].mean(axis=0) for k in range(len(unique))])
            ub_t = np.linalg.lstsq(unique, gv_mean, rcond=None)[0]
            if abs(np.linalg.det(ub_t)) < 1e-15:
                continue
            trial = np.linalg.inv(ub_t.T)
            if np.linalg.det(trial) <= 0:
                continue
            g = grain.grain(trial)
            cell_change = np.max(np.abs(g.unitcell / seed['grain'].unitcell - 1))
            if (misorientation_deg(g.U, seed['grain'].U) <= max_refine_angle_deg
                    and cell_change <= max_cell_relative_change):
                ubi = trial
    keep, hkls, err = support(data, xy, ubi, layer['y0'])
    n = int(keep.sum())
    nuniq = len(np.unique(hkls[keep], axis=0)) if n else 0
    nspots = (len(np.unique(data['spot3d_id'][keep]))
              if n and 'spot3d_id' in data else (n if kind == '4d' else None))
    bins = len(np.unique(np.floor((data['omega'][keep] % 360) / omega_bin_deg))) if n else 0
    # With no spot IDs, 2D row count cannot stand in for independent reflections.
    enough_spots = nspots is not None and nspots >= min_spots
    review = nuniq >= min_unique_hkl and bins >= min_omega_bins and enough_spots
    return dict(target=seed['target'], sources=seed['sources'], kind=kind, xy=xy,
                seed_xy=xy0, ubi=ubi, data=data, keep=keep, nrows=n, initial_nrows=int(initial),
                nuniq=nuniq, nspots=nspots, omega_bins=bins,
                hkl_rms=float(np.sqrt(np.mean(err[keep] ** 2))) if n else np.nan,
                review_candidate=bool(review),
                status='review' if review else ('no_spot_ids' if n and nspots is None else 'weak_or_absent'))

results = []
for ti in tqdm(sorted(set(s['target'] for s in seeds)), desc='Target layers'):
    local = [s for s in seeds if s['target'] == ti]
    for kind in ('4d', '2d'):
        peak_sets = local_peak_sets(stack[ti], local, kind)
        for seed, data in zip(local, peak_sets):
            results.append(fit_candidate(seed, data, stack[ti], kind))
    # Avoid retaining raw peak caches for every target layer.
    stack[ti]['ds']._pk2d = None
    stack[ti]['ds']._pk4d = None

for cid, result in enumerate(results):
    result['candidate_id'] = cid
summary = pd.DataFrame([dict(candidate_id=r['candidate_id'], target=stack[r['target']]['spec']['name'],
    pool=r['kind'], sources=r['sources'], sx=r['xy'][0], sy=r['xy'][1],
    initial_rows=r['initial_nrows'], matched_rows=r['nrows'], spots=r['nspots'],
    unique_hkl=r['nuniq'], omega_bins=r['omega_bins'], hkl_rms=r['hkl_rms'],
    status=r['status']) for r in results])
if len(summary):
    display(summary.sort_values(['unique_hkl', 'matched_rows'], ascending=False))
else:
    print('No absent-neighbour hypotheses; check neighbour and matching controls.')
plot_stack(stack, results)

## Review trajectories and reciprocal-lattice residuals

Choose a `candidate_id` from the table. Grey points passed the broad seed gate; coloured points pass the final orientation and trajectory gates. Inspect angular coverage and outliers, not just peak count. Red crosses in the stack mean “passes screening”, not “accepted”. A `no_spot_ids` 2D result stays unclassified because repeated frame observations are not independent reflections.

In [ ]:
def inspect_candidate(candidate_id):
    r = results[candidate_id]
    data, keep = r['data'], r['keep']
    if data is None:
        print('No local residual peaks for this seed and pool.')
        return
    layer = stack[r['target']]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
    stride = max(1, int(np.ceil(len(keep) / float(plot_peak_limit))))
    axes[0].scatter(data['omega'][::stride] % 360, data['dty'][::stride], s=2, c='0.8')
    axes[0].scatter(data['omega'][keep] % 360, data['dty'][keep], s=4, c='tab:blue')
    om = np.linspace(0, 360, 721)
    pred = trajectory(r['xy'], om, layer['y0'])
    axes[0].plot(om, pred, 'r-', label='Fitted COM')
    axes[0].fill_between(om, pred - trajectory_width, pred + trajectory_width, alpha=0.15)
    axes[0].plot(om, trajectory(r['seed_xy'], om, layer['y0']), 'k--', label='Source COM')
    axes[0].set(xlabel='omega modulo 360 (deg)', ylabel='dty')
    axes[0].legend()
    _, _, error = hkl_match(as_colfile(data), r['ubi'])
    axes[1].hist(error, bins=60, histtype='step', label='Local residual peaks')
    axes[1].hist(error[keep], bins=30, alpha=0.5, label='Matched')
    axes[1].axvline(hkl_tol, color='red')
    axes[1].set(xlabel='Distance to integer hkl', ylabel='Rows')
    axes[1].legend()
    axes[2].scatter(data['omega'][keep] % 360,
                    data['dty'][keep] - trajectory(r['xy'], data['omega'][keep], layer['y0']), s=4)
    axes[2].set(xlabel='omega modulo 360 (deg)', ylabel='dty - fitted trajectory')
    fig.suptitle('Candidate {} / {} / {} / {}'.format(candidate_id, layer['spec']['name'], r['kind'], r['status']))
    plt.show()

if results:
    best = max(results, key=lambda r: (r['nuniq'], r['nrows']))
    inspect_candidate(best['candidate_id'])
# inspect_candidate(0)

## Check competing claims and prepare grains in memory

The overlap table compares candidates **within the same target and peak pool**, using exact input row indices. A 2D and a 4D row index are different namespaces. Shared support, twins, and duplicate seeds must be resolved before acceptance. Cross-pool agreement can be assessed from fitted COM and orientation; do not add their counts.

Set `accepted_candidate_ids` after reviewing the plots. Choose one fit per recovered grain, normally the 2D fit if well supported. This creates `augmented_grains` in memory for a subsequent mapping run; it does not save or append to the dataset. Both existing and newly accepted grains are checked for duplicate COM/orientation matches.

In [ ]:
overlaps = []
for i, a in enumerate(results):
    if not a['nrows']:
        continue
    ra = a['data']['pool_row'][a['keep']]
    for b in results[i + 1:]:
        if b['target'] != a['target'] or b['kind'] != a['kind'] or not b['nrows']:
            continue
        shared = len(np.intersect1d(ra, b['data']['pool_row'][b['keep']]))
        if shared:
            overlaps.append(dict(candidate_a=a['candidate_id'], candidate_b=b['candidate_id'],
                                 shared_rows=shared, fraction_of_smaller=shared / float(min(a['nrows'], b['nrows']))))
display(pd.DataFrame(overlaps))

accepted_candidate_ids = []
augmented_grains = {layer['spec']['name']: list(layer['grains']) for layer in stack}
for cid in accepted_candidate_ids:
    r = results[cid]
    layer = stack[r['target']]
    name = layer['spec']['name']
    if not r['nrows']:
        raise ValueError('Cannot accept a candidate with no matched peaks.')
    new = grain.grain(r['ubi'].copy(), translation=np.r_[r['xy'], 0.0])
    if any(same_grain(new, r['xy'], old) for old in augmented_grains[name]):
        raise ValueError('Candidate {} duplicates a grain already in {}'.format(cid, name))
    # Reject unresolved row ownership among selected fits in the same pool.
    for other_id in accepted_candidate_ids:
        if other_id >= cid:
            continue
        other = results[other_id]
        if other['target'] == r['target'] and other['kind'] == r['kind'] and other['nrows']:
            shared = np.intersect1d(r['data']['pool_row'][r['keep']],
                                    other['data']['pool_row'][other['keep']])
            if len(shared):
                raise ValueError('Selected candidates share peaks; resolve ownership first.')
    augmented_grains[name].append(new)
print('In-memory grain counts:', {key: len(value) for key, value in augmented_grains.items()})

## Optional reconstruction from a candidate's matched 2D peaks

Choose a 2D result to reconstruct the local residual support using ImageD11's grain sinogram pipeline. This is additional spatial evidence, not independent validation. Reconstruction quality depends on actual angular coverage. The method uses the measured omega/dty coordinates through ImageD11, not a reshaping of acquisition frames. It may be unreliable for sparse or half scans; inspect the sinogram first.

In [ ]:
reconstruct_candidate_id = None  # set to a reviewed 2D candidate ID
if reconstruct_candidate_id is not None:
    r = results[reconstruct_candidate_id]
    if r['kind'] != '2d' or r['nrows'] < 20:
        raise ValueError('Select a 2D candidate with at least 20 contributing rows.')
    layer = stack[r['target']]
    g = grain.grain(r['ubi'].copy(), translation=np.r_[r['xy'], 0.0])
    gs = GrainSinogram(g, layer['ds'])
    gs.cf_for_sino = as_colfile({key: value[r['keep']] for key, value in r['data'].items()})
    gs.compute_hkl(store=True)
    gs.build_sinogram()
    from ImageD11.sinograms.geometry import sino_shift_and_pad
    shift, pad = sino_shift_and_pad(layer['y0'], len(layer['ds'].ybincens),
                                    layer['ds'].ymin, layer['ds'].ystep)
    gs.update_recon_parameters(pad=pad, shift=shift, y0=layer['y0'])
    gs.recon()
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    ax[0].pcolormesh(gs.sinoangles, layer['ds'].ybincens, gs.ssino, shading='auto')
    ax[0].set(xlabel='Projection angle (deg)', ylabel='dty', title='Matched residual sinogram')
    ax[1].imshow(gs.recons['iradon'], origin='lower')
    ax[1].set(xlabel='Reconstruction column', ylabel='Reconstruction row', title='Residual support reconstruction')
    plt.show()